# Eternal Dao — Episode 1 • One-click resumable illustrations

**Version 2 — corrected Colab dependencies and reduced Google sign-in.**

Run **Runtime → Restart session and run all** for the first corrected run. Later, **Run all** is enough. The notebook requests only the **Google Drive mount**, not a second Google Docs permission. You may save your own copy using **File → Save a copy in Drive**. Google may still require Drive authorization on a new runtime for security.

The approved script and storyboard were already placed **privately** in your existing Episode 1 Drive folder. Nothing from the novel is stored in public GitHub. This job produces about 201 individually saved 16:9 manhua PNGs, with automatic image approval, retry/recovery, progress logs and resume after disconnection.

**It never displays images and never compiles/publishes a video.** Free Colab GPU access and runtime longevity are not guaranteed.

In [ ]:
%pip -q install --upgrade "diffusers==0.35.2" "peft==0.17.1" "transformers==4.56.2" "accelerate>=1.2,<2" "huggingface_hub>=0.34,<1.0" "safetensors>=0.5" "google-api-python-client>=2.0"

In [ ]:
import hashlib, json, math, re, time, gc, traceback
from datetime import datetime, timezone
from pathlib import Path
from importlib.metadata import version
from packaging.version import Version
import torch

# Check the EXACT dependency that caused the previous failure *before*
# downloading any multi-gigabyte models or asking for Drive permission.
if Version(version("peft")) < Version("0.17.0"):
    raise RuntimeError("PEFT upgrade failed. Restart this runtime once and use Run all.")
if version("diffusers") != "0.35.2":
    raise RuntimeError("Diffusers version changed unexpectedly; rerun the dependency cell.")
# Import compatibility is checked BEFORE Drive login or large model downloads.
# If this old Colab runtime cached PEFT 0.10, restart it once to load 0.17.1.
try:
    import peft
    from peft import LoraConfig
    from diffusers import StableDiffusionXLPipeline, EulerDiscreteScheduler
    if Version(peft.__version__) < Version("0.17.0"):
        raise ImportError("The runtime cached an outdated PEFT module.")
    if not hasattr(StableDiffusionXLPipeline, "load_lora_weights"):
        raise ImportError("The SDXL pipeline has no LoRA loader.")
except (ImportError, AttributeError) as exc:
    raise RuntimeError("Image-library preflight failed; restart session once and Run all.") from exc
if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > T4 GPU before Run all.")
print("Dependency preflight PASSED: diffusers", version("diffusers"),
      "| peft", version("peft"), "| transformers", version("transformers"))
print("GPU:", torch.cuda.get_device_name(0),
      "| VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))

# One Google authorization flow, not separate Drive and Docs logins.
from google.colab import drive
MOUNT = Path("/content/drive")
if not (MOUNT / "MyDrive").is_dir():
    drive.mount(str(MOUNT))
BASE = (MOUNT / "MyDrive" / "Audiobook_Studio" /
        "Eternal Dao of the Lost Epoch — D03 — Illustration Library" /
        "Episode 01 — Chapters 1–2 — Original Illustrations")
SOURCE = BASE / "EP01_D03_APPROVED_SCRIPT.txt"
STORYBOARD = BASE / "EP01_D03_APPROVED_STORYBOARD.txt"
if not SOURCE.is_file() or not STORYBOARD.is_file():
    raise RuntimeError("Private approved source files missing in Drive folder. "
                       "Do not upload them publicly or use an alternative story.")
SCRIPT = SOURCE.read_text(encoding="utf-8-sig").replace("\r\n", "\n")
BOARD = STORYBOARD.read_text(encoding="utf-8-sig").replace("\r\n", "\n")
if (not SCRIPT.lstrip().startswith("CHAPTER ONE") or
        "CHAPTER TWO" not in SCRIPT or
        "The Wrong Sky" not in SCRIPT or
        "The Weight of an Old Name" not in SCRIPT or
        len(SCRIPT.split()) != 3613):
    raise RuntimeError("Private script does not match the approved D03 Chapters 1–2.")
if "SCENE-BY-SCENE" not in BOARD or "LOCKED CHARACTER SHEETS" not in BOARD:
    raise RuntimeError("Private D03 storyboard is incomplete.")
print("Approved private source loaded from Drive:", len(SCRIPT.split()), "words.",
      "No second Google Docs authorization.")

In [ ]:
# The 24 storyboard beats are not 24 pictures. Split the verbatim episode
# into many ~18-word visual shots, without modifying any spoken narration.
WORDS = SCRIPT.split()
BEATS = [{"id": int(i), "title": t.strip(), "visual": v.strip()}
         for i, t, v in re.findall(r"(?m)^\s*(\d{2})\s+([^\n]+)\nVisual:\s*([^\n]+)", BOARD)]
if len(BEATS) != 24 or [b["id"] for b in BEATS] != list(range(1, 25)):
    raise RuntimeError("The original approved 24 storyboard beats could not be read")
cast_block = BOARD.split("LOCKED CHARACTER SHEETS", 1)[1].split("SCENE-BY-SCENE", 1)[0]
CAST = dict((name.strip(), info.strip()) for name, info in
            re.findall(r"(?m)^([A-Z][A-Z\s]+)\s+[—–-]\s*([^\n]+)", cast_block))
chapter_two = next((i for i in range(len(WORDS)-1)
                    if WORDS[i].upper().strip(".,:") == "CHAPTER"
                    and WORDS[i+1].upper().strip(".,:") == "TWO"), None)
if chapter_two is None:
    raise RuntimeError("Cannot locate Chapter Two boundary")
SHOTS_N = math.ceil(len(WORDS) / 18)
SHA = hashlib.sha256(SCRIPT.encode()).hexdigest()
# Keep the ALREADY CREATED private D03 version directory rather than
# accidentally starting a second folder because Docs and text exports hash differently.
TARGET = BASE / "D03-612d3c8d39d1"
if not TARGET.parent.exists():
    raise RuntimeError("The existing private Google Drive illustration folder is unavailable")
IMAGES = TARGET / "illustrations"
IMAGES.mkdir(parents=True, exist_ok=True)
SHOTS = []
camera = ["wide establishing view", "medium character interaction", "expressive reaction close-up",
          "over-the-shoulder viewpoint", "detail of the relevant tool or object",
          "low architectural angle", "cinematic side profile", "tense three-quarter framing",
          "intimate dramatic close-up", "grounded physical action shot"]
style = ("Single full-frame 16:9 2D Chinese manhua illustration, original ink-defined "
         "drawn aesthetic, cinematic lighting, believable anatomy and practical technology "
         "blended with Chinese cultivation, dark brick city and copper industrial devices "
         "as appropriate to the narrated moment. NOT a storyboard collage. ")
exclude = ("No caption, no text, no watermarks, no extra panels, no dragons, no unexplained "
           "jade relics or mystical golden vortex, no floating fantasy pagodas, no unrelated "
           "sect battles or blood moon, no 3D render. ")
for index in range(SHOTS_N):
    start = index * len(WORDS) // SHOTS_N
    end = (index + 1) * len(WORDS) // SHOTS_N
    excerpt = " ".join(WORDS[start:end])
    nearby = " ".join(WORDS[max(0,start-20):min(len(WORDS),end+20)])
    if start < chapter_two:
        beat_index = min(11, int((start/max(1,chapter_two))*12))
    else:
        beat_index = 12 + min(11, int(((start-chapter_two)/max(1,len(WORDS)-chapter_two))*12))
    beat = BEATS[beat_index]
    context = (nearby + " " + beat["visual"]).lower()
    characters = []
    for name, description in CAST.items():
        aliases = [name.lower(), name.split()[-1].lower()]
        if any(len(a)>3 and re.search(r"\b"+re.escape(a)+r"\b", context) for a in aliases):
            characters.append(name + ": " + description)
    # Put the exact SCENE FIRST: CLIP truncates very long text prompts.
    # Keep physical action, world and cast within the short initial prompt.
    prompt = ("Original 2D Chinese manhua, cinematic 16:9. "
              + "Scene: " + beat["visual"][:145] + ". "
              + "Spoken moment: " + nearby[:140] + ". "
              + ("Appearance: " + " / ".join(characters[:2])[:110] + ". " if characters else "")
              + "Camera: " + camera[index % len(camera)] + ". "
              + "Grounded futuristic engineering with ancient cultivation, "
                "no captions, no panels, no dragons, no irrelevant fantasy palaces.")
    source_hash = hashlib.sha256(excerpt.encode()).hexdigest()
    prompt_hash = hashlib.sha256((prompt+source_hash).encode()).hexdigest()
    SHOTS.append({"index":index, "start_word":start,"end_word":end,
                  "storyboard_beat":beat["id"],"source_excerpt":excerpt,
                  "source_sha256":source_hash,"prompt":prompt,
                  "prompt_sha256":prompt_hash,
                  "file":f"scene-{index:04d}-{prompt_hash[:12]}.png"})
if " ".join(s["source_excerpt"] for s in SHOTS) != " ".join(WORDS):
    raise RuntimeError("Narration coverage is not exact")
def save_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(".part")
    temp.write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding="utf8")
    temp.replace(path)
save_json(TARGET/"PRIVATE_EP01_SHOT_PLAN.json",
          {"source_sha256":SHA,"storyboard_sha256":hashlib.sha256(BOARD.encode()).hexdigest(),
           "owner_artwork_approval":"blanket_preapproved","total_shots":len(SHOTS),
           "characters":CAST,"shots":SHOTS})
print("Prepared",len(SHOTS),"unique illustrations from",len(WORDS),"exact narration words")
print("Private Drive directory:",TARGET)

In [ ]:
# Retry-safe original image generation. No inline pictures.
STEPS, RETRIES_PER_IMAGE = 4, 3
MAX_NEW_IMAGES_THIS_SESSION = 0  # 0 = all missing images
from PIL import Image

def now_utc():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")

def valid_image(path):
    try:
        if path.stat().st_size < 12000:
            return False
        with Image.open(path) as pic:
            size, kind = pic.size, pic.format
            pic.verify()
        return size == (1024,576) and kind == "PNG"
    except (OSError, ValueError, FileNotFoundError):
        return False

def error_log(kind, index, attempt, error):
    with (TARGET/"PRIVATE_EP01_ERROR_LOG.jsonl").open("a",encoding="utf8") as out:
        out.write(json.dumps({"time":now_utc(),"kind":kind,"scene":index,
                              "attempt":attempt,"error":str(error)[:600]})+"\n")

def status(stage, failed=0, current=None):
    done = sum(valid_image(IMAGES/s["file"]) for s in SHOTS)
    data = {"state":stage,"updated_at":now_utc(),"episode":1,
            "source_sha256":SHA,"planned_images":len(SHOTS),
            "verified_images":done,"remaining_images":len(SHOTS)-done,
            "failed_this_run":failed,"current_scene":current,
            "owner_artwork_approval":"blanket_preapproved"}
    save_json(TARGET/"PRIVATE_EP01_STATUS.json",data)
    save_json(TARGET/"PRIVATE_EP01_PROGRESS.json",data)
    return data

def manifest_report():
    records=[]
    hashes=set()
    for s in SHOTS:
        image=IMAGES/s["file"]
        if valid_image(image):
            image_hash=hashlib.sha256(image.read_bytes()).hexdigest()
            if image_hash in hashes:
                raise RuntimeError("Duplicate image bytes: artwork must be distinct.")
            hashes.add(image_hash)
            records.append({"index":s["index"],"storyboard_beat":s["storyboard_beat"],
                            "file":"illustrations/"+s["file"],
                            "source_sha256":s["source_sha256"],
                            "image_sha256":image_hash})
    report={"schema":"private-episode-illustration-pack-v2","episode":1,
            "story_id":"eternal-dao-of-the-lost-epoch",
            "script_source_sha256":SHA,"owner_artwork_approval":"blanket_preapproved",
            "planned_images":len(SHOTS),"verified_images":len(records),
            "complete":len(records)==len(SHOTS),"compilation_allowed":False,
            "images":records,"updated_at":now_utc()}
    save_json(TARGET/"PRIVATE_EP01_ILLUSTRATION_MANIFEST.json",report)
    done_file=TARGET/"PRIVATE_EP01_DONE.json"
    if report["complete"]:
        save_json(done_file,{"status":"COMPLETE","updated_at":now_utc(),
                             "script_source_sha256":SHA,
                             "verified_images":len(records)})
    else:
        done_file.unlink(missing_ok=True)
    return report

existing=sum(valid_image(IMAGES/s["file"]) for s in SHOTS)
print("Verified images already stored:", existing, "/", len(SHOTS))
status("PREFLIGHT_OK")
failure_count, generated = 0, 0
if existing < len(SHOTS):
    status("LOADING_MODEL")
    try:
        from diffusers import StableDiffusionXLPipeline, EulerDiscreteScheduler
        from huggingface_hub import hf_hub_download
        print("Loading SDXL (existing downloads cached in this Colab runtime)")
        pipe=StableDiffusionXLPipeline.from_pretrained(
            "stabilityai/stable-diffusion-xl-base-1.0",
            torch_dtype=torch.float16,variant="fp16",use_safetensors=True)
        ckpt=hf_hub_download("ByteDance/SDXL-Lightning",
                             f"sdxl_lightning_{STEPS}step_lora.safetensors")
        pipe.load_lora_weights(ckpt)
        pipe.fuse_lora()
        pipe.scheduler=EulerDiscreteScheduler.from_config(
            pipe.scheduler.config,timestep_spacing="trailing")
        pipe.to("cuda")
        pipe.enable_vae_slicing()
        pipe.set_progress_bar_config(disable=True)
    except Exception as exc:
        error_log("MODEL_LOAD",None,1,exc)
        status("FAILED_MODEL_LOAD")
        raise
    status("GENERATING")
    started=time.monotonic()
    for shot in SHOTS:
        dest=IMAGES/shot["file"]
        if valid_image(dest):
            continue
        if MAX_NEW_IMAGES_THIS_SESSION and generated>=MAX_NEW_IMAGES_THIS_SESSION:
            break
        success=False
        for attempt in range(1,RETRIES_PER_IMAGE+1):
            temp=dest.with_suffix(".partial.png")
            try:
                temp.unlink(missing_ok=True)
                with torch.inference_mode():
                    picture=pipe(shot["prompt"],num_inference_steps=STEPS,
                                 guidance_scale=0.0,width=1024,height=576,
                                 generator=torch.Generator("cuda").manual_seed(
                                     580000+shot["index"]*113+attempt-1)).images[0]
                picture.save(temp,format="PNG")
                if not valid_image(temp):
                    raise RuntimeError("Generated PNG not valid or wrong resolution")
                temp.replace(dest)
                generated+=1
                success=True
                break
            except Exception as exc:
                temp.unlink(missing_ok=True)
                error_log("IMAGE",shot["index"],attempt,exc)
                gc.collect()
                torch.cuda.empty_cache()
                if attempt<RETRIES_PER_IMAGE:
                    time.sleep(5*attempt)
        if not success:
            failure_count+=1
        if generated==1 or generated%5==0 or not success:
            info=status("GENERATING",failed=failure_count,current=shot["index"])
            if generated and generated%10==0:
                print("Saved",info["verified_images"],"/",len(SHOTS),
                      "| sec/image",round((time.monotonic()-started)/generated,1),
                      "| failures",failure_count)
    del pipe
    torch.cuda.empty_cache()
report=manifest_report()
status("COMPLETE" if report["complete"] else "INCOMPLETE",failed=failure_count)
print("Verified private Drive PNGs:",report["verified_images"],"/",report["planned_images"])
print("Run all again to resume missing scenes when incomplete; nothing displayed.")